
The implementation sequence is:

1. baseline detailed-model inference;
2. reduced-model inference;
3. behavioral-support GP learning with GP-MPM;
4. behaviorally guided detailed-model inference using the learned GP through `custom_joint_prior`.


Make sure that the following exist for the detailed model in your notebook: 

detailed_param_names (a list of names of the two parameters), 

detailed_prior (a dictionary with names as keys and list for values),

detailed_lower_bounds (array), 

detailed_upper_bounds (array), 

a function that generates the detailed_trajectory_fn by taking beta values, 

a function that returns the detailed_log_likelihood by computing the L2 norm of the model and the data,

Then, for the reduced model, make sure that the following  exist:

reduced_param_names

reduced_prior

reduced_log_likelihood


### Detailed model

```python
detailed_param_names = ["beta1", "beta2"]

detailed_prior = {
    "beta1": [...],
    "beta2": [...],
}

detailed_lower_bounds = np.array([...])
detailed_upper_bounds = np.array([...])

def detailed_trajectory_fn(params_dict):
    # Returns the detailed trajectory in the same representation
    # used by the reduced model and observed data.
    ...

def detailed_log_likelihood(params_dict, data):
    # Returns one scalar log likelihood.
    ...
```

### Reduced model

```python
reduced_param_names = ["alpha"]

reduced_prior = {
    "alpha": [...],
}

def reduced_trajectory_fn(params_dict):
    # Returns the reduced trajectory in the same representation
    # used by the detailed model and observed data.
    ...

def reduced_log_likelihood(params_dict, data):
    # Returns one scalar log likelihood.
    ...
```

### Shared data

For the toy problem, a simple structure is enough:

```python
t = ...
y_obs = ...

data = {
    "t": t,
    "y_obs": y_obs,
}
```

The trajectory functions should return compatible output shapes, typically `(len(t),)` for this toy example.


### `posterior_level_set_boundary`

This is the behavioral-support level

\gamma \in (0,1),

which defines

$K_{\epsilon,\gamma}
=
\{\theta:b_\epsilon(\theta)\ge\gamma\}$

The GP learns the boundary

$b_\epsilon(\theta)=\gamma$

### `data_discrepancy_threshold`

This is the detailed-model data-fit tolerance

\eta,

which defines

$F_\eta=\{\theta:d(y(\theta),\bar y)\le\eta\}$

### `epsilon`

This is the finite-horizon behavioral compatibility tolerance. A reduced-posterior sample supports a detailed parameter when

\$d\!\left(y(\theta),\widehat y(\widehat\theta)\right)\le \epsilon$

### `gp_screening_level`

This is


\zeta,


the GP screening probability level. During behaviorally guided detailed-model MCMC, a proposal is screened when

$p_K(\theta)=\Phi\!\left(\frac{\mu(\theta)}{\sigma(\theta)}\right)\le \zeta$

In [ ]:
from implementation_methods_theory_updated import (
    py_inference,
    build_behavioral_gp_from_reduced_posterior,
)


## Quick-run settings for MCMC

In [ ]:
# MCMC settings
NWALKERS = 16
NSTEPS = 1000
BURN = 200
RANDOM_SEED = 2026

# Behavioral-support / GP thresholds
epsilon = ...
posterior_level_set_boundary = ...   # gamma
data_discrepancy_threshold = ...     # eta
gp_screening_level = 0.01            # zeta

# Small GP-MPM budget for a quick end-to-end run
N_INITIAL_GP = 6
N_MPM_STEPS = 10
N_INTEGRATION_POINTS = 200
N_ACQUISITION_CANDIDATES = 50


# 1. Baseline detailed-model inference

This gives the ordinary detailed-model posterior without behavioral screening.

In [ ]:
detailed_result = py_inference(
    likelihood=detailed_log_likelihood,
    data=data,
    params_to_estimate=detailed_param_names,
    prior=detailed_prior,
    nwalkers=NWALKERS,
    nsteps=NSTEPS,
    discard=BURN,
    random_seed=RANDOM_SEED,
)


The primary posterior sample array is:

```python
detailed_result.flat_samples
```

For later comparison, the full native `emcee` chain is also available as:

```python
detailed_result.chain
```

# 2. Reduced-model inference

Run the reduced-model posterior inference using the same `py_inference` interface.

In [ ]:
reduced_result = py_inference(
    likelihood=reduced_log_likelihood,
    data=data,
    params_to_estimate=reduced_param_names,
    prior=reduced_prior,
    nwalkers=NWALKERS,
    nsteps=NSTEPS,
    discard=BURN,
    random_seed=RANDOM_SEED,
)


For the GP-learning step, use the post-burn-in reduced posterior samples:

```python
reduced_result.flat_samples
```

# 3. Learn the behavioral-support level set with GP-MPM

The high-level wrapper below:

1. takes the reduced posterior samples;
2. generates/stores the corresponding reduced trajectories;
3. evaluates the Monte Carlo behavioral support
   \(\widehat b_{\epsilon,M}(\theta)\) at selected detailed parameters;
4. trains the GP on
   \(\widehat b_{\epsilon,M}(\theta)-\gamma\);
5. sequentially adds detailed evaluations using GP-MPM;
6. returns a frozen GP object suitable for use as a `custom_joint_prior`.

For the toy model, `detailed_log_scale=False` is usually the natural choice when \(\beta_1,\beta_2\) are bounded initial conditions on a linear parameter domain.

In [ ]:
gp_result = build_behavioral_gp_from_reduced_posterior(
    reduced_result.flat_samples,

    reduced_param_names=reduced_param_names,
    reduced_trajectory_fn=reduced_trajectory_fn,

    detailed_param_names=detailed_param_names,
    detailed_lower_bounds=detailed_lower_bounds,
    detailed_upper_bounds=detailed_upper_bounds,
    detailed_trajectory_fn=detailed_trajectory_fn,

    observed_trajectory=y_obs,

    epsilon=epsilon,
    posterior_level_set_boundary=posterior_level_set_boundary,
    data_discrepancy_threshold=data_discrepancy_threshold,
    gp_screening_level=gp_screening_level,

    detailed_log_scale=False,

    n_initial=N_INITIAL_GP,
    n_mpm_steps=N_MPM_STEPS,
    n_integration_points=N_INTEGRATION_POINTS,
    n_acquisition_candidates=N_ACQUISITION_CANDIDATES,

    random_seed=RANDOM_SEED,
)


The two main returned objects are:

```python
gp_result.validity_model
gp_result.custom_joint_prior
```

Useful stored quantities include:

```python
gp_result.gp_mpm.X_train
gp_result.gp_mpm.parameters_train
gp_result.gp_mpm.behavioral_support_train
gp_result.gp_mpm.validity_train
gp_result.reduced_posterior_fit_mass
```



# 4. Behaviorally guided detailed-model inference

Run the detailed model again with the learned, frozen GP supplied through `custom_joint_prior`.

The ordinary detailed-model prior and likelihood remain unchanged.

In [ ]:
detailed_gp_result = py_inference(
    likelihood=detailed_log_likelihood,
    data=data,
    params_to_estimate=detailed_param_names,
    prior=detailed_prior,
    custom_joint_prior=gp_result.custom_joint_prior,
    nwalkers=NWALKERS,
    nsteps=NSTEPS,
    discard=BURN,
    random_seed=RANDOM_SEED,
)


# Minimal outputs from the quick run

After the four blocks execute, the main objects are:

```python
# Baseline detailed posterior
detailed_result.flat_samples

# Reduced posterior
reduced_result.flat_samples

# Learned behavioral GP
gp_result.validity_model

# Behaviorally guided detailed posterior
detailed_gp_result.flat_samples
```

For the later full experiment, these are the two detailed-model posterior sample sets to compare:

```python
samples_baseline = detailed_result.flat_samples
samples_guided = detailed_gp_result.flat_samples
```